<a href="https://colab.research.google.com/github/dev-08/prompt-engineering-chatbot/blob/main/Ai_chatbot_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Simple OpenAI Chatbot in Google Colab

This notebook shows how to:

- install the OpenAI Python SDK
- configure an API key in Colab
- make a first API call
- improve responses with prompt engineering
- build a simple multi-turn chatbot

The goal is to understand the **engineering flow** behind a basic LLM application:
**user input → prompt/message history → OpenAI API call → model response → chat memory**


## 1. Install the OpenAI SDK

This installs the Python package used to send requests to the OpenAI API.


In [ ]:
# Install the official OpenAI Python SDK inside the Colab runtime.
!pip install -q openai


## 2. Set the API key

The SDK reads the API key from the `OPENAI_API_KEY` environment variable.

**Important:** never upload a notebook to GitHub with a real API key inside it.
Before publishing, replace the real key with a placeholder like `"your_api_key_here"`.


In [ ]:
import os

# Store your OpenAI API key in an environment variable so the SDK can read it.
# Replace the placeholder string with your real key only in your private notebook.
os.environ["OPENAI_API_KEY"] = "your_api_key_here"


## 3. Create the OpenAI client

The client object is the main interface used to call the OpenAI API from Python.


In [ ]:
from openai import OpenAI

# Create an API client.
# It automatically uses the OPENAI_API_KEY environment variable above.
client = OpenAI()


## 4. First API call

This is the simplest example: send one prompt to the model and print the response.

### Engineering idea
At this stage, there is no conversation memory yet.
The model only sees the single input string passed in this request.


In [ ]:
# Send a simple one-shot request to the model.
response = client.responses.create(
    model="gpt-5.4",
    input="Explain what a large language model is in simple words."
)

# output_text contains the final plain-text response returned by the model.
print(response.output_text)


## 5. Intro to prompt engineering

Prompt engineering means writing better instructions so the model behaves the way you want.

In this example, the request includes:

- a **developer** message that defines the assistant's role and style
- a **user** message that asks the actual question

This usually gives better results than a short raw prompt because the model has clearer guidance.


In [ ]:
# Here we use structured messages instead of a plain string.
# The developer message acts like behavior/configuration for the assistant.
response = client.responses.create(
    model="gpt-5.4",
    input=[
        {
            "role": "developer",
            "content": (
                "You are a beginner-friendly AI tutor. "
                "Explain clearly, use simple English, and give short examples."
            )
        },
        {
            "role": "user",
            "content": "What is prompt engineering?"
        }
    ]
)

print(response.output_text)


## 6. Build a simple chatbot

Now the notebook keeps a running `conversation` list.

### Why this matters
The model does **not automatically remember** past turns across separate API calls.
Your Python code is responsible for preserving chat history and sending it back with each new request.

That is the core engineering idea behind a simple multi-turn chatbot.


In [ ]:
# The conversation list stores the full chat history.
# The first message defines the default assistant behavior.
conversation = [
    {
        "role": "developer",
        "content": "You are a helpful beginner-friendly chatbot. Keep answers clear and simple."
    }
]

def chat(user_message):
    """
    Send one user message to the model while preserving prior conversation history.

    Engineering flow:
    1. Add the new user message to the conversation list
    2. Send the full conversation to the API
    3. Read the model's reply
    4. Save that reply back into the conversation list
    5. Return the reply for display
    """
    global conversation

    # Append the latest user input so the model can see it.
    conversation.append({
        "role": "user",
        "content": user_message
    })

    # Send the full chat history, not just the latest message.
    response = client.responses.create(
        model="gpt-5.4",
        input=conversation
    )

    # Extract the plain-text reply from the API response.
    reply = response.output_text

    # Save the assistant response so future turns include this context too.
    conversation.append({
        "role": "assistant",
        "content": reply
    })

    return reply


## 7. Test the chatbot

This first test asks a simple beginner question.


In [ ]:
# First multi-turn test message
print(chat("Hi, what is an LLM?"))


## 8. Continue the conversation

This second question depends on the previous answer.
Because the full `conversation` list is sent again, the model can answer in context.


In [ ]:
# Follow-up question using previous context from the conversation list
print(chat("How is it different from traditional machine learning?"))


## Next steps

Good improvements after this notebook:

1. add error handling for invalid keys or quota issues
2. stream tokens for a better chat experience
3. trim old messages to control token cost
4. add a small UI with Gradio or Streamlit
5. connect a RAG pipeline for external knowledge
